# 05 — Use a saved adapted model
This performs prediction only. It does not upload models or send texts to a hosted model endpoint. The whole original label space remains available.

In [1]:
from pathlib import Path
import os, sys
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/lid_finetuning_bundle')]
ROOT = next((p for p in candidates if (p/'config.json').exists() and (p/'lidlab').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the complete ZIP first and set ROOT to its lid_finetuning_bundle folder.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Project folder:', ROOT)


Project folder: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method


In [2]:
from lidlab.data import load_config, clean_text
from lidlab.backends import NativeFastText, ConLIDBackend
c = load_config()
MODEL = 'nllb'  # 'nllb', 'glotlid', or 'conlid'
PHASE = 'replay'  # or 'target_only'
folder = ROOT/c['output_dir']/MODEL/PHASE
if MODEL == 'conlid':
    model = ConLIDBackend(folder, c['device'])
else:
    model = NativeFastText(folder/'model.bin', ROOT)
print('Number of supported output labels:', len(model.labels))

Number of supported output labels: 220


In [3]:
texts = ['සිංහල භාෂාව ශ්‍රී ලංකාවේ භාවිත වේ.', 'This is a short English example.']
texts = [clean_text(t, c['normalization']) for t in texts]
labels, probabilities = model.predict(texts, ROOT/'inference_temp')
for text, label, probability in zip(texts, labels, probabilities):
    print(text, '->', label, round(probability, 4))

සිංහල භාෂාව ශ්‍රී ලංකාවේ භාවිත වේ. -> __label__sin_Sinh 0.9711
This is a short English example. -> __label__eng_Latn 1.0


For NLLB/GlotLID, the generated `model.bin` also loads using the standard `fasttext.load_model()` API once a compatible fastText Python package is installed. For ConLID, preserve `model.safetensors`, `config.json`, `labels.json`, and `vocab.json` together. Its checkpoint retains the released tensor names and can be loaded by the official mean-pooling inference architecture.

Before publishing a model on Hugging Face, include upstream attribution, data provenance, licenses, the full label list, results, and the exact inference instructions. This notebook does not perform publication.